In [0]:
"""
Configuration File for Banking Agent Project
============================================

This file contains all settings for the banking agent.
Change values here instead of editing code everywhere.

Key concepts:
- WORKSPACE_PATH: Where files are stored in Databricks
- CHUNK_SIZE: How big each policy chunk is
- EMBEDDING_MODEL: Which model creates embeddings
- DATABASE: Where vectors are stored
"""

import os

# ============================================
# 1. FILE PATHS (WHERE DATA IS STORED)
# ============================================

# Your Databricks workspace folder path
WORKSPACE_USER = "molugurikoushik68@gmail.com"
WORKSPACE_PATH = f"/Workspace/Users/{WORKSPACE_USER}/banking-agent"

# Data folder paths
DATA_PATH = f"{WORKSPACE_PATH}/data"
POLICIES_FILE = f"{DATA_PATH}/banking_policies.txt"
TRANSACTIONS_FILE = f"{DATA_PATH}/sample_transactions.csv"
EMBEDDINGS_FILE = f"{DATA_PATH}/embeddings.parquet"

# Model paths (where we save trained models)
MODELS_PATH = f"{WORKSPACE_PATH}/models"

print("✓ File paths configured:")
print(f"  Data path: {DATA_PATH}")
print(f"  Policies: {POLICIES_FILE}")
print(f"  Transactions: {TRANSACTIONS_FILE}")

# ============================================
# 2. DOCUMENT PROCESSING SETTINGS
# ============================================

# How to split policies into chunks
CHUNK_SIZE = 150 # Words per chunk (roughly 2000 characters)
CHUNK_OVERLAP = 50 # Words that overlap between chunks (for context)

# Why these values?
# - 500 words = ~2000 characters (good size for LLM)
# - 100 word overlap = preserves context between chunks
# Example:
#   Chunk 1: [words 0-500]
#   Chunk 2: [words 400-900]  (100 word overlap with Chunk 1)
#   This way, no information is lost at chunk boundaries

print("✓ Chunking configured:")
print(f"  Chunk size: {CHUNK_SIZE} words")
print(f"  Overlap: {CHUNK_OVERLAP} words")

# ============================================
# 3. EMBEDDING & VECTOR SEARCH SETTINGS
# ============================================

# Which Databricks model to use for embeddings
# Options: "gte-large-en", "bge-base-en", "gte-small-en"
# We use gte-large-en (good quality, reasonable size)
EMBEDDING_MODEL = "databricks-bge-large-en"

# Embedding dimension (how many numbers represent each text)
# gte-large-en creates 1024-dimensional vectors
# More dimensions = better quality but slower/more expensive
EMBEDDING_DIMENSION = 1024

# Why embeddings?
# Text "What is transfer limit?" → [0.123, 0.456, ..., 0.789] (1024 numbers)
# Numbers represent "meaning" of text
# Similar text → similar numbers
# This lets us find policies by meaning, not just keywords

print("✓ Embedding configured:")
print(f"  Model: {EMBEDDING_MODEL}")
print(f"  Dimension: {EMBEDDING_DIMENSION}")

# ============================================
# 4. VECTOR SEARCH SETTINGS
# ============================================

# Where to store embeddings (Databricks table)
CATALOG = "main"  # Default catalog
SCHEMA = "banking_agent_db"  # Your database
EMBEDDINGS_TABLE = "policy_embeddings"  # Table name

# Full table path
EMBEDDINGS_TABLE_PATH = f"{CATALOG}.{SCHEMA}.{EMBEDDINGS_TABLE}"

# How many similar policies to return when searching
TOP_K = 3  # Return top 3 most relevant policies

# Minimum similarity score to consider a result relevant
MIN_SIMILARITY = 0.5  # (0.0 = completely different, 1.0 = identical)

print("✓ Vector search configured:")
print(f"  Table: {EMBEDDINGS_TABLE_PATH}")
print(f"  Return top K: {TOP_K}")
print(f"  Min similarity: {MIN_SIMILARITY}")

# ============================================
# 5. BANKING CONSTRAINTS & SECURITY
# ============================================

# Transfer limits (fraud prevention)
MAX_TRANSFER_AMOUNT = 10000  # Max per transaction
DAILY_TRANSFER_LIMIT = 25000  # Max per day
TRANSFER_CONFIRMATION_THRESHOLD = 1000  # Require confirmation above this

# Rate limiting (prevent abuse)
RATE_LIMIT_PER_MINUTE = 10  # Max requests per minute per user

# Fraud detection settings
FRAUD_SCORE_THRESHOLD = 0.5  # Flag if fraud probability > 50%

print("✓ Banking constraints configured:")
print(f"  Max transfer: ${MAX_TRANSFER_AMOUNT}")
print(f"  Daily limit: ${DAILY_TRANSFER_LIMIT}")
print(f"  Fraud threshold: {FRAUD_SCORE_THRESHOLD}")

# ============================================
# 6. MODEL SETTINGS
# ============================================

# LLM settings (for agent reasoning)
LLM_MODEL = "databricks-dbrx-instruct"  # Databricks foundation model
LLM_TEMPERATURE = 0  # 0 = deterministic (important for banking)
LLM_MAX_TOKENS = 500  # Max response length

# Why temperature = 0?
# Temperature controls randomness:
# - 0.0 = always same answer (good for banking)
# - 1.0 = random variations (good for creative)
# Banking needs consistency, so we use 0

print("✓ LLM configured:")
print(f"  Model: {LLM_MODEL}")
print(f"  Temperature: {LLM_TEMPERATURE}")

# ============================================
# 7. DATABASE SETUP
# ============================================

def setup_database():
    """
    Create Databricks database if it doesn't exist.
    Called once at startup.
    """
    try:
        # Create schema (database)
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
        print(f"✓ Database schema created: {SCHEMA}")
    except Exception as e:
        print(f"✗ Error creating schema: {e}")

# ============================================
# 8. FUNCTION: Get Config Dictionary
# ============================================

def get_config():
    """
    Returns all configuration as a dictionary.
    Useful for passing settings to functions.
    
    Example:
        config = get_config()
        chunk_size = config['chunk_size']
    """
    return {
        # File paths
        'workspace_path': WORKSPACE_PATH,
        'data_path': DATA_PATH,
        'policies_file': POLICIES_FILE,
        'transactions_file': TRANSACTIONS_FILE,
        'embeddings_file': EMBEDDINGS_FILE,
        
        # Chunking
        'chunk_size': CHUNK_SIZE,
        'chunk_overlap': CHUNK_OVERLAP,
        
        # Embeddings
        'embedding_model': EMBEDDING_MODEL,
        'embedding_dimension': EMBEDDING_DIMENSION,
        
        # Vector search
        'catalog': CATALOG,
        'schema': SCHEMA,
        'embeddings_table': EMBEDDINGS_TABLE,
        'embeddings_table_path': EMBEDDINGS_TABLE_PATH,
        'top_k': TOP_K,
        'min_similarity': MIN_SIMILARITY,
        
        # Banking
        'max_transfer': MAX_TRANSFER_AMOUNT,
        'daily_limit': DAILY_TRANSFER_LIMIT,
        'fraud_threshold': FRAUD_SCORE_THRESHOLD,
        
        # LLM
        'llm_model': LLM_MODEL,
        'llm_temperature': LLM_TEMPERATURE,
        'llm_max_tokens': LLM_MAX_TOKENS,
    }

# ============================================
# 9. VERIFICATION
# ============================================

print("\n" + "="*60)
print("CONFIGURATION SUMMARY")
print("="*60)
print(f"✓ Workspace: {WORKSPACE_PATH}")
print(f"✓ Data files: {DATA_PATH}")
print(f"✓ Embedding model: {EMBEDDING_MODEL}")
print(f"✓ Vector table: {EMBEDDINGS_TABLE_PATH}")
print(f"✓ LLM model: {LLM_MODEL}")
print("="*60)
print("✓ Configuration loaded successfully!")
print("="*60)

# Setup database
setup_database()
